# Quit-Risk Model From Presence History

This notebook builds an early-warning model from the daily presence workbook.

Important limitation:
- The workbook contains only **1 explicit `Sortie`** event.
- Because of that, we do **not** have enough clean quit labels to train a reliable classical attrition model.
- The target used below is a **proxy**: `target_quit_14d = 1` when an employee has an explicit exit in the next 14 days **or** disappears from the file within the next 14 days and then stays inactive for at least 14 more days.

So this notebook should be used as a **risk scoring / early warning** workflow, not as a final HR decision system.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

DATA_PATH = Path(r"f:\Downloads\presence_history_jan_to_8_avril_fixed_mise_en_demeure_logic.xlsx")
SHEET_NAME = "detail"
PREDICTION_HORIZON_DAYS = 14
INACTIVE_GAP_DAYS = 14
RANDOM_STATE = 42

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Excel file not found: {DATA_PATH}")


In [ ]:
raw_df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

standard_columns = [
    "employee_id",
    "employee_name",
    "group",
    "phone",
    "cc",
    "function_sap",
    "hire_date",
    "anciennete_raw",
    "supervisor",
    "segment",
    "gender",
    "nature",
    "entry_exit",
    "presence_hours",
    "motif",
    "eff_active",
    "eff_present",
    "eff_mr",
    "abs_p_per",
    "abs_np",
    "hours_sup",
    "late_count",
    "day",
    "month_name",
]

if len(raw_df.columns) != len(standard_columns):
    raise ValueError(f"Unexpected number of columns: {len(raw_df.columns)}")

rename_map = dict(zip(raw_df.columns, standard_columns))
df = raw_df.rename(columns=rename_map).copy()

month_map = {
    "Janvier": 1,
    "Fevrier": 2,
    "F?vrier": 2,
    "Mars": 3,
    "Avril": 4,
}

df["date"] = pd.to_datetime(
    {
        "year": 2026,
        "month": df["month_name"].map(month_map),
        "day": pd.to_numeric(df["day"], errors="coerce"),
    },
    errors="coerce",
)
df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce")
df["anciennete_years"] = (
    df["anciennete_raw"].astype(str).str.replace(",", ".", regex=False)
)
df["anciennete_years"] = pd.to_numeric(df["anciennete_years"], errors="coerce")
df = df.sort_values(["employee_id", "date"]).reset_index(drop=True)

DATASET_END = df["date"].max()

print(f"Rows: {len(df):,}")
print(f"Employees: {df['employee_id'].nunique():,}")
print(f"Observation window: {df['date'].min().date()} -> {DATASET_END.date()}")
print("Entry/exit events:")
print(df["entry_exit"].value_counts(dropna=False))

display(df.head())


In [ ]:
employee_summary = (
    df.groupby("employee_id")
    .agg(
        employee_name=("employee_name", "first"),
        first_seen=("date", "min"),
        last_seen=("date", "max"),
        explicit_exit=("entry_exit", lambda s: (s == "Sortie").any()),
        explicit_entry=("entry_exit", lambda s: (s == "Entr?e").any()),
        rows=("employee_id", "size"),
    )
    .reset_index()
)
employee_summary["days_before_dataset_end"] = (DATASET_END - employee_summary["last_seen"]).dt.days

print("Employees with explicit Sortie:", int(employee_summary["explicit_exit"].sum()))
print("Employees not seen for 14+ days before dataset end:", int((employee_summary["days_before_dataset_end"] >= INACTIVE_GAP_DAYS).sum()))

display(employee_summary.sort_values(["explicit_exit", "days_before_dataset_end"], ascending=[False, False]).head(15))


## Snapshot Training Logic

Instead of training one row per employee, we create one row per **employee snapshot date**.
For each snapshot date we compute rolling features from the previous 14 days, then label whether that employee will:

1. have an explicit `Sortie` in the next 14 days, or
2. disappear from the file within the next 14 days and stay inactive long enough to confirm the disappearance.

That turns the problem into: **"Based on the recent attendance pattern, is this employee likely to leave soon?"**


In [ ]:
def trailing_streak(values, predicate):
    streak = 0
    for value in reversed(values):
        if predicate(value):
            streak += 1
        else:
            break
    return streak


def build_feature_row(employee_history, snapshot_date):
    history = employee_history.loc[employee_history["date"] <= snapshot_date].tail(14).copy()

    motifs = history["motif"].fillna("")
    hours = pd.to_numeric(history["presence_hours"], errors="coerce").fillna(0)
    late = pd.to_numeric(history["late_count"], errors="coerce").fillna(0)
    overtime = pd.to_numeric(history["hours_sup"], errors="coerce").fillna(0)
    abs_np = pd.to_numeric(history["abs_np"], errors="coerce").fillna(0)
    abs_p = pd.to_numeric(history["abs_p_per"], errors="coerce").fillna(0)
    mr = pd.to_numeric(history["eff_mr"], errors="coerce").fillna(0)

    first_row = employee_history.iloc[0]

    return {
        "employee_id": first_row["employee_id"],
        "employee_name": first_row["employee_name"],
        "group": first_row["group"],
        "function_sap": first_row["function_sap"],
        "supervisor": first_row["supervisor"],
        "segment": first_row["segment"],
        "gender": first_row["gender"],
        "nature": first_row["nature"],
        "anciennete_years": first_row["anciennete_years"],
        "snapshot_date": snapshot_date,
        "days_since_start": int((snapshot_date - employee_history["date"].min()).days),
        "history_days": int(len(history)),
        "hours_sum_14d": float(hours.sum()),
        "hours_mean_14d": float(hours.mean()),
        "zero_hour_days_14d": int((hours == 0).sum()),
        "low_hour_days_14d": int((hours <= 4).sum()),
        "overtime_sum_14d": float(overtime.sum()),
        "delay_count_14d": float(late.sum()),
        "abs_np_sum_14d": float(abs_np.sum()),
        "abs_p_sum_14d": float(abs_p.sum()),
        "mr_sum_14d": float(mr.sum()),
        "mise_en_demeure_days_14d": int((motifs == "Mise en demeure").sum()),
        "sans_questionnaire_days_14d": int((motifs == "Sans questionnaire").sum()),
        "conge_sans_solde_days_14d": int((motifs == "Conge sans solde").sum()),
        "absence_autorise_days_14d": int((motifs == "Absence Autorise").sum()),
        "maladie_prolongee_days_14d": int((motifs == "Maladie prolongee").sum()),
        "recent_zero_streak": int(trailing_streak(hours.tolist(), lambda x: x == 0)),
        "recent_mise_en_demeure_streak": int(
            trailing_streak(motifs.tolist(), lambda x: x == "Mise en demeure")
        ),
        "current_month": int(snapshot_date.month),
    }


def build_training_snapshots(data, prediction_horizon_days=14, inactive_gap_days=14):
    rows = []

    for employee_id, employee_history in data.groupby("employee_id"):
        employee_history = employee_history.sort_values("date").copy()
        last_seen = employee_history["date"].max()
        explicit_exit_dates = set(
            employee_history.loc[employee_history["entry_exit"] == "Sortie", "date"]
        )

        for snapshot_date in employee_history["date"]:
            future_limit = snapshot_date + pd.Timedelta(days=prediction_horizon_days)

            # Drop snapshots that are too close to the dataset end because their future is censored.
            if future_limit > DATASET_END - pd.Timedelta(days=inactive_gap_days):
                continue

            explicit_exit_soon = any(snapshot_date < d <= future_limit for d in explicit_exit_dates)
            disappears_within_horizon = last_seen <= future_limit
            inactivity_is_confirmed = last_seen <= DATASET_END - pd.Timedelta(days=inactive_gap_days)

            row = build_feature_row(employee_history, snapshot_date)
            row["target_quit_14d"] = int(
                explicit_exit_soon or (disappears_within_horizon and inactivity_is_confirmed)
            )
            rows.append(row)

    return pd.DataFrame(rows)


def build_latest_snapshots(data):
    rows = []

    for employee_id, employee_history in data.groupby("employee_id"):
        employee_history = employee_history.sort_values("date").copy()
        rows.append(build_feature_row(employee_history, employee_history["date"].max()))

    latest = pd.DataFrame(rows)
    latest["days_since_last_seen"] = (DATASET_END - latest["snapshot_date"]).dt.days
    latest["is_recently_active"] = latest["days_since_last_seen"] <= 7
    return latest


In [ ]:
snapshot_df = build_training_snapshots(
    df,
    prediction_horizon_days=PREDICTION_HORIZON_DAYS,
    inactive_gap_days=INACTIVE_GAP_DAYS,
)

print(f"Training snapshots: {len(snapshot_df):,}")
print(snapshot_df["target_quit_14d"].value_counts())
print(f"Positive rate: {snapshot_df['target_quit_14d'].mean():.2%}")

display(
    snapshot_df.loc[snapshot_df["target_quit_14d"] == 1, [
        "employee_id",
        "employee_name",
        "snapshot_date",
        "mise_en_demeure_days_14d",
        "recent_mise_en_demeure_streak",
        "abs_np_sum_14d",
        "zero_hour_days_14d",
    ]].head(10)
)


In [ ]:
feature_columns = [
    c
    for c in snapshot_df.columns
    if c not in ["employee_id", "employee_name", "snapshot_date", "target_quit_14d"]
]
X = snapshot_df[feature_columns].copy()
y = snapshot_df["target_quit_14d"].copy()
groups = snapshot_df["employee_id"].copy()

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.50).astype(int)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Test positives: {int(y_test.sum())}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"Average precision: {average_precision_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual_stay", "actual_quit_proxy"],
    columns=["pred_stay", "pred_quit_proxy"],
)
display(cm)


In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients),
})

print("Top features pushing risk UP:")
display(coef_df.sort_values("coefficient", ascending=False).head(12))

print("Top features pushing risk DOWN:")
display(coef_df.sort_values("coefficient", ascending=True).head(12))


In [ ]:
final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)
final_model.fit(X, y)

latest_df = build_latest_snapshots(df)
latest_df["quit_risk_probability"] = final_model.predict_proba(latest_df[feature_columns])[:, 1]

recent_risk_table = latest_df.loc[latest_df["is_recently_active"]].sort_values(
    "quit_risk_probability", ascending=False
)

print("Latest risk view for recently active employees (last seen within 7 days):")
display(
    recent_risk_table[[
        "employee_id",
        "employee_name",
        "snapshot_date",
        "days_since_last_seen",
        "quit_risk_probability",
        "recent_mise_en_demeure_streak",
        "mise_en_demeure_days_14d",
        "abs_np_sum_14d",
        "zero_hour_days_14d",
        "hours_mean_14d",
    ]]
)


## Next Improvement Ideas

1. Replace the proxy target with true HR leaver labels from payroll, resignation, or dismissal records.
2. Add more months of history so the model learns real seasonality and longer behavior patterns.
3. Add team-level context such as shift, plant area, workload, and supervisor changes.
4. Calibrate the score before production use and choose a threshold based on business cost, not just 0.50.
